<h1>Important</h1>

- The following notebook has only personal learning purposes with no further intention. This was developed using AI tools combined with multiple iterations to refine the code given at first it does generate many errors, from documentation error and more.
- The key intention of this notebook is to show to use a model available from Hugging Face in combination with techniques from the same library to define a LLM from scratch and apply some fine tuning techniques.
- This is not a comercial or industry code to be used, it is just a personal academic learning of how to use different python libraries with ideas of Reinforcement Learning and LLMs.

Key considerations when running this:
- The machine that was used had installed cuda nvidia with 8 GB of capacity, and the idea was to constraint the dataset size and memory usage when training the model.
- Do not load more than 1 model if the CUDA capacity is small because it will lead to potential crushing.
- Try to use cuda and not cpu because is much faster when running.

_____

In [1]:
# Complete code form model LLM and trainig
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
import numpy as np
import math
import gc
import warnings
from typing import Optional, List, Union, Dict
import time
import os
import json
from datasets import Dataset
from tokenizers import Tokenizer, models, pre_tokenizers, trainers, decoders
from transformers import PreTrainedTokenizerFast
from trl import SFTTrainer, SFTConfig
import uuid

warnings.filterwarnings("ignore")

# ==========================
# Step 1: Memory Management Utilities
# ==========================
# These functions help monitor and manage GPU memory usage, critical for running on an 8GB CUDA GPU.


def cleanup_memory():
    """Clears unused variables and GPU cache to free memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def get_memory_info():
    """Returns current GPU memory usage in GB or CPU mode if no GPU is available."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        return f"GPU: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved"
    return "CPU mode"


# Set device (CUDA if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Initial memory: {get_memory_info()}")

# ==========================
# Step 2: Model Configuration
# ==========================
# Defines the configuration for the language model, optimized for an 8GB GPU with balanced parameters.


class LLMConfig:
    """Configuration for the LLM, optimized for 8GB GPU."""

    vocab_size = 10000  # Vocabulary size for the tokenizer
    d_model = 512  # Hidden dimension size for embeddings and layers
    n_heads = 8  # Number of attention heads
    n_layers = 8  # Number of transformer layers
    max_seq_len = 1024  # Maximum sequence length for input/output
    dropout = 0.085  # Dropout rate to prevent overfitting
    learning_rate = 8e-5  # Learning rate for pretraining
    batch_size = 6  # Batch size for memory efficiency
    gradient_accumulation_steps = 4  # Accumulate gradients over steps
    warmup_steps = 100  # Warmup steps for learning rate scheduler
    max_steps = 5000  # Total training steps
    pad_token_id = 0  # ID for padding token
    eos_token_id = 1  # ID for end-of-sequence token
    bos_token_id = 2  # ID for beginning-of-sequence token
    unk_token_id = 3  # ID for unknown token


config = LLMConfig()

# Estimate model size for debugging
estimated_params = (
    config.vocab_size * config.d_model  # Token embeddings
    + config.n_layers
    * (
        4 * config.d_model * config.d_model  # Attention weights
        + 3 * config.d_model * config.d_model  # Feed-forward
    )
) / 1e6
print(f"Estimated model parameters: {estimated_params:.1f}M")
print(f"Estimated memory usage: {estimated_params * 4:.0f}MB (fp32)")

# ==========================
# Step 3: Load Training Dataset
# ==========================
# Loads the initial training dataset from a text file for pretraining.


def load_training_data(file_path):
    """Loads training texts from a text file, one text per line."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            texts = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(texts)} training texts from {file_path}")
        print(
            f"Average text length: {np.mean([len(text) for text in texts]):.0f} characters"
        )
        return texts
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return []


training_texts = load_training_data("training_data.txt")
if not training_texts:
    raise ValueError(
        "No training data loaded. Please provide a valid training_data.txt file."
    )

# ==========================
# Step 4: Tokenizer Setup
# ==========================
# Creates and trains a Byte-Pair Encoding (BPE) tokenizer using HuggingFace's tokenizers library.
# Reference: HuggingFace Tokenizers (https://huggingface.co/docs/tokenizers/en/index)

print("Creating BPE tokenizer...")
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=config.vocab_size,
    special_tokens=["<pad>", "</s>", "<s>", "<unk>"],
    min_frequency=2,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)


def get_training_corpus():
    """Yields training texts for tokenizer training."""
    for text in training_texts:
        yield text


tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
tokenizer.enable_padding(pad_id=0, pad_token="<pad>")

fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    pad_token="<pad>",
    eos_token="</s>",
    bos_token="<s>",
    unk_token="<unk>",
    model_max_length=config.max_seq_len,
    padding_side="right",
)

fast_tokenizer.save_pretrained("./bpe_tokenizer")

# Test tokenizer
test_text = "Hello! How are you learning mathematics today?"
encoded = fast_tokenizer.encode(test_text)
decoded = fast_tokenizer.decode(encoded)
print(f"\nTokenizer Test:")
print(f"Original: '{test_text}'")
print(f"Encoded: {encoded} ({len(encoded)} tokens)")
print(f"Decoded: '{decoded}'")
print(f"Memory after tokenizer: {get_memory_info()}")

# ==========================
# Step 5: Transformer Components
# ==========================
# Defines core transformer components: normalization, positional embeddings, attention, and feed-forward layers.


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization for stable training.
    Normalizes input by its root mean square, improving training stability."""

    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        return self.scale * x / rms


class RotaryPositionalEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE) to encode token positions in attention.
    Applies rotation to query and key vectors based on position.
    Reference: RoFormer (https://arxiv.org/abs/2104.09864)"""

    def __init__(self, dim, max_seq_len=8192, base=10000.0):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, seq_len, device):
        t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_pos_emb(q, k, cos, sin):
    """Applies rotary positional embeddings to query and key tensors."""

    def rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention mechanism for capturing token relationships.
    Splits input into multiple heads, applies scaled dot-product attention, and combines outputs.
    """

    def __init__(self, config):
        super().__init__()
        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.d_k = config.d_model // config.n_heads
        assert (
            config.d_model % config.n_heads == 0
        ), "d_model must be divisible by n_heads"
        self.w_q = nn.Linear(config.d_model, config.d_model, bias=False)
        self.w_k = nn.Linear(config.d_model, config.d_model, bias=False)
        self.w_v = nn.Linear(config.d_model, config.d_model, bias=False)
        self.w_o = nn.Linear(config.d_model, config.d_model, bias=False)
        self.rope = RotaryPositionalEmbedding(self.d_k, config.max_seq_len)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, attention_mask=None):
        original_shape = x.shape
        if len(x.shape) == 4:
            x = x.squeeze(1)
        if len(x.shape) != 3:
            raise ValueError(f"Expected 3D tensor after reshape, got shape {x.shape}")
        batch_size, seq_len, d_model = x.shape
        Q = (
            self.w_q(x)
            .view(batch_size, seq_len, self.n_heads, self.d_k)
            .transpose(1, 2)
        )
        K = (
            self.w_k(x)
            .view(batch_size, seq_len, self.n_heads, self.d_k)
            .transpose(1, 2)
        )
        V = (
            self.w_v(x)
            .view(batch_size, seq_len, self.n_heads, self.d_k)
            .transpose(1, 2)
        )
        cos, sin = self.rope(seq_len, x.device)
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        Q, K = apply_rotary_pos_emb(Q, K, cos, sin)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device), diagonal=1
        ).bool()
        scores.masked_fill_(causal_mask, float("-inf"))
        if attention_mask is not None:
            attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(~attention_mask.bool(), float("-inf"))
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context = torch.matmul(attn_weights, V)
        context = (
            context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        )
        output = self.w_o(context)
        if len(original_shape) == 4:
            output = output.unsqueeze(1)
        return output


class SwiGLU(nn.Module):
    """SwiGLU activation for feed-forward networks, combining SiLU and gating.
    Reference: GLU Variants (https://arxiv.org/abs/2002.05202)"""

    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = int(8 * d_model / 3)
            d_ff = ((d_ff + 255) // 256) * 256
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        gate = F.silu(self.w1(x))
        value = self.w2(x)
        return self.w3(gate * value)


# Test transformer components
print("Testing transformer components...")
test_input = torch.randn(1, 16, config.d_model, device=device)
test_mask = torch.ones(1, 16, device=device)
rms_norm = RMSNorm(config.d_model).to(device)
attention = MultiHeadAttention(config).to(device)
swiglu = SwiGLU(config.d_model).to(device)
with torch.no_grad():
    norm_out = rms_norm(test_input)
    attn_out = attention(test_input, test_mask)
    ff_out = swiglu(test_input)
print(f"✓ RMSNorm: {test_input.shape} -> {norm_out.shape}")
print(f"✓ Attention: {test_input.shape} -> {attn_out.shape}")
print(f"✓ SwiGLU: {test_input.shape} -> {ff_out.shape}")
print(f"Memory usage: {get_memory_info()}")
del rms_norm, attention, swiglu, test_input, test_mask, norm_out, attn_out, ff_out
cleanup_memory()

# ==========================
# Step 6: Transformer Layer and Model
# ==========================
# Combines components into a transformer layer and full language model.


class TransformerLayer(nn.Module):
    """A single transformer layer with pre-norm architecture.
    Applies attention, normalization, and feed-forward in sequence."""

    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(config)
        self.feed_forward = SwiGLU(config.d_model)
        self.norm1 = RMSNorm(config.d_model)
        self.norm2 = RMSNorm(config.d_model)

    def forward(self, x, attention_mask=None):
        attn_out = self.attention(self.norm1(x), attention_mask)
        x = x + attn_out
        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out
        return x


class ModelOutput:
    """HuggingFace-compatible output class to store logits and loss."""

    def __init__(self, logits, loss=None):
        self.logits = logits
        self.loss = loss

    def __getitem__(self, key):
        if isinstance(key, int):
            if key == 0:
                return self.loss
            elif key == 1:
                return self.logits
            raise IndexError(f"Index {key} out of range")
        elif isinstance(key, str):
            return getattr(self, key)
        raise TypeError(f"Key must be int or str, got {type(key)}")

    def __contains__(self, key):
        return hasattr(self, key)

    def keys(self):
        keys_list = []
        if hasattr(self, "logits") and self.logits is not None:
            keys_list.append("logits")
        if hasattr(self, "loss") and self.loss is not None:
            keys_list.append("loss")
        return keys_list


class ModernLLM(nn.Module):
    """Complete transformer-based language model.
    Combines token embeddings, transformer layers, and a language modeling head."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList(
            [TransformerLayer(config) for _ in range(config.n_layers)]
        )
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.gradient_checkpointing = False
        self.apply(self._init_weights)
        print(
            f"Model created with {sum(p.numel() for p in self.parameters()):,} parameters"
        )

    def _init_weights(self, module):
        """Initializes weights with normal distribution for better convergence."""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self, input_ids, attention_mask=None, labels=None, use_cache=None, **kwargs
    ):
        """Forward pass through the model, computing logits and optional loss."""
        x = self.token_embedding(input_ids)
        if attention_mask is None:
            attention_mask = (input_ids != self.config.pad_token_id).long()
        for layer in self.layers:
            if self.gradient_checkpointing and self.training:

                def create_custom_forward(module):
                    def custom_forward(*inputs):
                        return module(*inputs)

                    return custom_forward

                x = checkpoint(
                    create_custom_forward(layer), x, attention_mask, use_reentrant=False
                )
            else:
                x = layer(x, attention_mask)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            loss_fn = nn.CrossEntropyLoss(ignore_index=self.config.pad_token_id)
            loss = loss_fn(shift_logits, shift_labels)
        return ModelOutput(logits=logits, loss=loss)

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        self.gradient_checkpointing = True
        print("Gradient checkpointing enabled.")

    def gradient_checkpointing_disable(self):
        self.gradient_checkpointing = False
        print("Gradient checkpointing disabled.")

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens=500,
        temperature=0.10,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.225,
    ):
        """Generates text by sampling from the model's output distribution."""
        self.eval()
        batch_size = input_ids.size(0)
        past_tokens = torch.zeros(
            (batch_size, config.vocab_size), device=input_ids.device
        )
        for _ in range(max_new_tokens):
            if input_ids.size(1) >= self.config.max_seq_len:
                input_ids = input_ids[:, -self.config.max_seq_len + 1 :]
            outputs = self(input_ids)
            logits = outputs.logits[:, -1, :] / temperature
            logits = logits - repetition_penalty * past_tokens
            if top_k > 0:
                top_k_logits, _ = torch.topk(logits, top_k)
                logits[logits < top_k_logits[:, [-1]]] = float("-inf")
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(
                    F.softmax(sorted_logits, dim=-1), dim=-1
                )
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[
                    ..., :-1
                ].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(
                    1, sorted_indices, sorted_indices_to_remove
                )
                logits[indices_to_remove] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            past_tokens.scatter_add_(
                1, next_token, torch.ones_like(next_token, dtype=logits.dtype)
            )
            if next_token.item() == fast_tokenizer.eos_token_id:
                break
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids


# Create and test model
print("Creating the complete LLM...")
print(f"Memory before model creation: {get_memory_info()}")
model = ModernLLM(config).to(device)
print(f"Memory after model creation: {get_memory_info()}")
print("\nTesting model forward pass...")
test_tokens = torch.randint(4, 100, (1, 20), device=device)
test_attention = torch.ones_like(test_tokens)
with torch.no_grad():
    outputs = model(test_tokens, attention_mask=test_attention, labels=test_tokens)
    print(f"✓ Forward pass successful")
    print(f"✓ Logits shape: {outputs.logits.shape}")
    print(f"✓ Loss: {outputs.loss:.4f}")
print(f"Memory after testing: {get_memory_info()}")

# ==========================
# Step 7: Training Utilities
# ==========================
# Utilities for creating batches and scheduling the learning rate.


def create_training_batch(texts, tokenizer, max_length=512, batch_size=2):
    """Creates training batches with padding and truncation."""
    input_ids_list = []
    attention_mask_list = []
    selected_texts = np.random.choice(
        texts, size=min(batch_size, len(texts)), replace=False
    )
    for text in selected_texts:
        encoding = tokenizer(
            text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )
        input_ids = encoding["input_ids"].squeeze(0).to(device)
        attention_mask = encoding["attention_mask"].squeeze(0).to(device)
        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
    return torch.stack(input_ids_list), torch.stack(attention_mask_list)


class LearningRateScheduler:
    """Schedules learning rate with warmup and cosine decay."""

    def __init__(self, optimizer, warmup_steps, max_steps, max_lr=5e-4):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.max_lr = max_lr
        self.current_step = 0

    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            lr = self.max_lr * self.current_step / self.warmup_steps
        else:
            progress = (self.current_step - self.warmup_steps) / (
                self.max_steps - self.warmup_steps
            )
            lr = self.max_lr * 0.5 * (1 + math.cos(math.pi * progress))
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr
        self.optimizer.step()
        return lr

    def zero_grad(self):
        self.optimizer.zero_grad()


# ==========================
# Step 8: Model Training
# ==========================
# Trains the model using gradient accumulation for memory efficiency.


def train_model(model, tokenizer, train_texts, config):
    """Trains the model with memory-efficient techniques."""
    print("Starting model training...")
    print(f"Training texts: {len(train_texts)}")
    print(f"Batch size: {config.batch_size}")
    print(f"Gradient accumulation steps: {config.gradient_accumulation_steps}")
    print(
        f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}"
    )
    model.train()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=0.01, eps=1e-8
    )
    scheduler = LearningRateScheduler(
        optimizer=optimizer,
        warmup_steps=config.warmup_steps,
        max_steps=config.max_steps,
        max_lr=config.learning_rate,
    )
    total_loss = 0.0
    log_interval = 25
    best_loss = float("inf")
    print(f"\nStarting training loop for {config.max_steps} steps...")
    print(f"Initial memory: {get_memory_info()}")
    for step in range(config.max_steps):
        input_ids, attention_mask = create_training_batch(
            train_texts,
            tokenizer,
            max_length=config.max_seq_len,
            batch_size=config.batch_size,
        )
        outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss / config.gradient_accumulation_steps
        loss.backward()
        total_loss += loss.item() * config.gradient_accumulation_steps
        if (step + 1) % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            current_lr = scheduler.step()
            scheduler.zero_grad()
        if (step + 1) % log_interval == 0:
            avg_loss = total_loss / log_interval
            perplexity = math.exp(min(avg_loss, 10))
            print(
                f"Step {step+1:4d}/{config.max_steps} | Loss: {avg_loss:.4f} | PPL: {perplexity:.2f} | LR: {current_lr:.2e} | {get_memory_info()}"
            )
            if avg_loss < best_loss:
                best_loss = avg_loss
                print(f"  ✓ New best loss: {best_loss:.4f}")
            total_loss = 0.0
        if (step + 1) % 100 == 0:
            cleanup_memory()
        if torch.cuda.is_available():
            memory_used = torch.cuda.memory_allocated() / 1024**3
            if memory_used > 7.0:
                print(f"⚠️ High memory usage: {memory_used:.2f}GB")
                cleanup_memory()
    print(f"\n✓ Training completed!")
    print(f"✓ Final loss: {best_loss:.4f}")
    print(f"✓ Final memory: {get_memory_info()}")
    return model


print("=" * 60)
print("TRAINING THE MODEL FROM SCRATCH")
print("=" * 60)
trained_model = train_model(model, fast_tokenizer, training_texts, config)

# ==========================
# Step 9: Initial Model Testing
# ==========================
# Tests the model’s generation capabilities before fine-tuning.


def test_initial_model(temperature=0.2):
    """Tests the pretrained model with general prompts."""
    test_prompts = [
        "Mathematics is the foundation of",
        "To solve a linear equation, you",
        "Physics explains the behavior of",
        "Chemistry is the study of",
        "Biology investigates how",
        "Calculus is essential for",
        "Statistics helps us make",
        "Computer science enables",
        "Quantum mechanics reveals",
        "The universe is governed by",
        "Data science provides insights",
        "Thermodynamics governs energy",
    ]
    print("Testing initial model generation...")
    trained_model.eval()
    for i, prompt in enumerate(test_prompts, 1):
        print(f"\nTest {i}: '{prompt}'")
        encoding = fast_tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = encoding["input_ids"]
        with torch.no_grad():
            generated = trained_model.generate(
                input_ids,
                max_new_tokens=500,
                temperature=temperature,
                top_k=50,
                top_p=0.95,
                repetition_penalty=1.225,
            )
        full_text = fast_tokenizer.decode(generated[0], skip_special_tokens=True)
        print(f"Generated: {full_text}")


test_initial_model()
cleanup_memory()

# ==========================
# Step 10: Load SFT Dataset
# ==========================
# Loads the Supervised Fine-Tuning dataset from a JSON file.


def load_sft_data(file_path):
    """Loads SFT instruction data from a JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            instruction_data = json.load(f)
        dataset = Dataset.from_list(instruction_data)
        print(f"SFT dataset prepared with {len(dataset)} examples")
        print(
            f"Average example length: {np.mean([len(ex['text']) for ex in instruction_data]):.0f} characters"
        )
        return dataset
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return Dataset.from_dict({"text": []})


sft_dataset = load_sft_data("sft_data.json")
if len(sft_dataset) == 0:
    raise ValueError("No SFT data loaded. Please provide a valid sft_data.json file.")

# ==========================
# Step 11: Model Wrapper
# ==========================
# Wraps the model for HuggingFace compatibility.


class HuggingFaceModelWrapper:
    """Wraps the model for compatibility with HuggingFace's SFTTrainer."""

    def __init__(self, model, tokenizer, config):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config

    def save_pretrained(self, save_directory):
        os.makedirs(save_directory, exist_ok=True)
        torch.save(
            self.model.state_dict(), os.path.join(save_directory, "pytorch_model.bin")
        )
        config_dict = {
            "vocab_size": self.config.vocab_size,
            "hidden_size": self.config.d_model,
            "num_hidden_layers": self.config.n_layers,
            "num_attention_heads": self.config.n_heads,
            "max_position_embeddings": self.config.max_seq_len,
            "model_type": "custom_llm",
            "architectures": ["ModernLLM"],
        }
        with open(os.path.join(save_directory, "config.json"), "w") as f:
            json.dump(config_dict, f, indent=2)
        self.tokenizer.save_pretrained(save_directory)
        print(f"Model and tokenizer saved to {save_directory}")


model_wrapper = HuggingFaceModelWrapper(trained_model, fast_tokenizer, config)

# ==========================
# Step 12: SFT Training
# ==========================
# Configures and runs Supervised Fine-Tuning.


def create_sft_training_config():
    """Creates SFT configuration optimized for 8GB GPU."""
    return {
        "output_dir": "./sft_model",
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "learning_rate": 4e-5,
        "num_train_epochs": 200,
        "max_length": 400,
        "logging_steps": 25,
        "save_steps": 100,
        "warmup_steps": 50,
        "weight_decay": 0.01,
        "fp16": torch.cuda.is_available(),
        "remove_unused_columns": False,
        "dataloader_drop_last": True,
        "gradient_checkpointing": True,
        "dataloader_num_workers": 0,
        "dataset_text_field": "text",
        "save_total_limit": 1,
        "save_strategy": "steps",
        "lr_scheduler_type": "cosine",
        "optim": "adamw_torch",
    }


print(f"Model parameters: {sum(p.numel() for p in trained_model.parameters()):,}")
print(f"SFT dataset size: {len(sft_dataset)}")
print(f"Memory before SFT setup: {get_memory_info()}")

model_path = "./base_trained_model"
model_wrapper.save_pretrained(model_path)

sft_config_dict = create_sft_training_config()
print("\nSFT Configuration:")
for key, value in sft_config_dict.items():
    print(f"  {key}: {value}")

if not hasattr(trained_model.config, "_name_or_path"):
    trained_model.config._name_or_path = "ModernLLM-custom"
if not hasattr(trained_model.config, "_attn_implementation"):
    trained_model.config._attn_implementation = "eager"

trained_model.gradient_checkpointing_enable()
sft_trainer = SFTTrainer(
    model=trained_model,
    args=SFTConfig(**sft_config_dict),
    train_dataset=sft_dataset,
    processing_class=fast_tokenizer,
)
sft_trainer.train()
final_model_path = "./sft_model_final"
sft_trainer.save_model(final_model_path)
cleanup_memory()

# ==========================
# Step 13: Enhanced Validation
# ==========================
# Tests the fine-tuned model with diverse, challenging prompts across multiple domains.


def validate_sft_model():
    """Validates the SFT model with diverse prompts in math, philosophy, science, and more."""
    validation_prompts = [
        {
            "prompt": "Human: Solve the quadratic equation x² - 5x + 6 = 0.",
            "category": "Mathematics",
            "expected": "Factorize or use the quadratic formula to find x = 2, x = 3.",
        },
        {
            "prompt": "Human: What is the ethical principle of utilitarianism?",
            "category": "Philosophy",
            "expected": "Utilitarianism is the principle that actions are right if they promote the greatest happiness for the greatest number.",
        },
        {
            "prompt": "Human: Explain the greenhouse effect in simple terms.",
            "category": "Environmental Science",
            "expected": "The greenhouse effect is when gases in Earth's atmosphere trap heat from the sun, warming the planet.",
        },
        {
            "prompt": "Human: What is the significance of the Turing Test in AI?",
            "category": "Computer Science",
            "expected": "The Turing Test evaluates if a machine can exhibit human-like intelligence by conversing indistinguishably from a human.",
        },
        {
            "prompt": "Human: How does natural selection drive evolution?",
            "category": "Biology",
            "expected": "Natural selection favors organisms with traits better suited to their environment, leading to changes in species over time.",
        },
        {
            "prompt": "Human: What is the derivative of f(x) = 3x² + 2x + 1?",
            "category": "Calculus",
            "expected": "The derivative is f'(x) = 6x + 2.",
        },
        {
            "prompt": "Human: Discuss the concept of free will versus determinism.",
            "category": "Philosophy",
            "expected": "Free will suggests individuals control their actions, while determinism argues actions are predetermined by prior causes.",
        },
        {
            "prompt": "Human: What is the primary source of energy for Earth's climate system?",
            "category": "Earth Science",
            "expected": "The Sun is the primary source of energy for Earth's climate system.",
        },
    ]
    print("=" * 50)
    print("VALIDATING SFT MODEL")
    print("=" * 50)
    trained_model.eval()
    for i, test_case in enumerate(validation_prompts, 1):
        print(f"\n{i}. Category: {test_case['category']}")
        print(f"Prompt: {test_case['prompt']}")
        print(f"Expected: {test_case['expected']}")
        print("-" * 40)
        encoding = fast_tokenizer(
            test_case["prompt"],
            return_tensors="pt",
            truncation=True,
            max_length=config.max_seq_len,
        ).to(device)
        input_ids = encoding["input_ids"]
        with torch.no_grad():
            generated = trained_model.generate(
                input_ids,
                max_new_tokens=500,
                temperature=0.1,
                top_k=50,
                top_p=0.95,
                repetition_penalty=1.225,
            )
        full_response = fast_tokenizer.decode(generated[0], skip_special_tokens=True)
        response_part = full_response[len(test_case["prompt"]) :].strip()
        print(f"Assistant: {response_part}")


validate_sft_model()

Using device: cuda
Initial memory: GPU: 0.00GB allocated, 0.00GB reserved
Estimated model parameters: 19.8M
Estimated memory usage: 79MB (fp32)
Loaded 1139 training texts from training_data.txt
Average text length: 160 characters
Creating BPE tokenizer...

Tokenizer Test:
Original: 'Hello! How are you learning mathematics today?'
Encoded: [43, 449, 82, 4, 948, 376, 1020, 1067, 947, 628, 2402, 4943, 34] (13 tokens)
Decoded: 'Hello! How are you learning mathematics today?'
Memory after tokenizer: GPU: 0.00GB allocated, 0.00GB reserved
Testing transformer components...
✓ RMSNorm: torch.Size([1, 16, 512]) -> torch.Size([1, 16, 512])
✓ Attention: torch.Size([1, 16, 512]) -> torch.Size([1, 16, 512])
✓ SwiGLU: torch.Size([1, 16, 512]) -> torch.Size([1, 16, 512])
Memory usage: GPU: 0.02GB allocated, 0.03GB reserved
Creating the complete LLM...
Memory before model creation: GPU: 0.01GB allocated, 0.02GB reserved
Model created with 37,511,680 parameters
Memory after model creation: GPU: 0.15GB a

Adding EOS to train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Gradient checkpointing enabled.


Step,Training Loss
25,61.243200
50,51.725800
75,45.080800
100,40.054400
125,35.769400
150,32.036100
175,28.925600
200,26.112300
225,23.706400
250,21.498600


VALIDATING SFT MODEL

1. Category: Mathematics
Prompt: Human: Solve the quadratic equation x² - 5x + 6 = 0.
Expected: Factorize or use the quadratic formula to find x = 2, x = 3.
----------------------------------------
Assistant: Assistant: Itaphat is the art?

Assistant: It involvesUsent distance that - x = 1)

*Inke: It'sri'xseleanning A (5 (5 (1. Thead-19b 2(? Subsen 5) Im?
- **sen/ 2x² - Gat seb Subsen ATrasteoridaeces: Acalidalence, time, and 2(Eemarrealence, merges, or questions, and po d in change, reproreal throughes, mwor-30ers include probability.

2. Category: Philosophy
Prompt: Human: What is the ethical principle of utilitarianism?
Expected: Utilitarianism is the principle that actions are right if they promote the greatest happiness for the greatest number.
----------------------------------------
Assistant: Assistant: The internet is a global network connecting billions of computers. It lets devices communicate worldwide through protocols like HTTP. Think of it as digit

In [2]:
# Step 14: Conversational Test with History
# ==========================
# Simulates a conversation with history to test context retention.


def test_conversation_with_history():
    """Tests the model with a simulated conversation history."""
    conversation_history = [
        {
            "user": "Human: What is gravity?",
            "assistant": "Gravity is the force that attracts objects toward each other, causing them to come together or move closer. On Earth, it pulls objects downward, making things fall.",
        },
        {
            "user": "Human: Can you explain how gravity affects planetary orbits?",
            "assistant": "Gravity keeps planets in orbit around the Sun by providing the centripetal force needed to maintain their elliptical paths. The Sun’s mass creates a gravitational field that pulls planets toward it, balancing their tangential velocity.",
        },
        {
            "user": "Human: How does this relate to Einstein’s theory of relativity?",
            "assistant": "Einstein’s general relativity describes gravity as the curvature of spacetime caused by mass. Planets follow geodesic paths in this curved spacetime, which explains their orbits more accurately than Newton’s laws, especially for strong gravitational fields.",
        },
    ]
    print("=" * 50)
    print("CONVERSATIONAL TEST WITH HISTORY")
    print("=" * 50)
    trained_model.eval()
    context = ""
    for turn in conversation_history:
        context += f"{turn['user']} {turn['assistant']} "
    final_prompt = "Human: How does relativity affect our understanding of black holes?"
    full_prompt = context + final_prompt
    print(f"Conversation History + Prompt:\n{full_prompt}")
    print("-" * 40)
    encoding = fast_tokenizer(
        full_prompt, return_tensors="pt", truncation=True, max_length=config.max_seq_len
    ).to(device)
    input_ids = encoding["input_ids"]
    with torch.no_grad():
        generated = trained_model.generate(
            input_ids,
            max_new_tokens=500,
            temperature=0.1,
            top_k=50,
            top_p=0.95,
            repetition_penalty=1.225,
        )
    full_response = fast_tokenizer.decode(generated[0], skip_special_tokens=True)
    response_part = full_response[len(full_prompt) :].strip()
    print(f"Assistant: {response_part}")


test_conversation_with_history()
cleanup_memory()

CONVERSATIONAL TEST WITH HISTORY
Conversation History + Prompt:
Human: What is gravity? Gravity is the force that attracts objects toward each other, causing them to come together or move closer. On Earth, it pulls objects downward, making things fall. Human: Can you explain how gravity affects planetary orbits? Gravity keeps planets in orbit around the Sun by providing the centripetal force needed to maintain their elliptical paths. The Sun’s mass creates a gravitational field that pulls planets toward it, balancing their tangential velocity. Human: How does this relate to Einstein’s theory of relativity? Einstein’s general relativity describes gravity as the curvature of spacetime caused by mass. Planets follow geodesic paths in this curved spacetime, which explains their orbits more accurately than Newton’s laws, especially for strong gravitational fields. Human: How does relativity affect our understanding of black holes?
----------------------------------------
Assistant: **fs tec

___

References

In [3]:
# Citations
print(
    """
CITATIONS AND ACKNOWLEDGMENTS

HuggingFace Tokenizers: https://huggingface.co/docs/tokenizers/en/index
RoFormer (Rotary Positional Embeddings): https://arxiv.org/abs/2104.09864
SwiGLU Activation: https://arxiv.org/abs/2002.05202
PyTorch Documentation: https://pytorch.org/docs/stable/index.html
Transformers Library: https://huggingface.co/docs/transformers/index
TRL (SFTTrainer): https://huggingface.co/docs/trl/en/index

This implementation is for educational purposes and builds upon the research and development 
efforts of the machine learning community. All libraries and models are used according 
to their respective licenses.
"""
)


CITATIONS AND ACKNOWLEDGMENTS

HuggingFace Tokenizers: https://huggingface.co/docs/tokenizers/en/index
RoFormer (Rotary Positional Embeddings): https://arxiv.org/abs/2104.09864
SwiGLU Activation: https://arxiv.org/abs/2002.05202
PyTorch Documentation: https://pytorch.org/docs/stable/index.html
Transformers Library: https://huggingface.co/docs/transformers/index
TRL (SFTTrainer): https://huggingface.co/docs/trl/en/index

This implementation is for educational purposes and builds upon the research and development 
efforts of the machine learning community. All libraries and models are used according 
to their respective licenses.

